In [4]:
import sys
import torch
import triton
import triton.language as tl
import os, json


@triton.jit
def mat_vec_kernel(
    vec_ptr,
    matrix_ptr,
    out_ptr,
    vec_stridex,
    matrix_stridey,
    matrix_stridex,
    out_stridex,
    K,
    BLOCK_SIZE_M: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE_M
    offsets = block_start + tl.arange(0, BLOCK_SIZE_M) # each process data in groups of block_size
    mask = offsets < K

    out_ptr = out_ptr + out_stridex * tl.arange(0, BLOCK_SIZE_M)
    #out_ptr = tl.reshape(out_ptr, (BLOCK_SIZE_M, 1)32

    matrix_x = block_start + matrix_stridex * tl.arange(0, BLOCK_SIZE_M)
    matrix_y = block_start + matrix_stridey * tl.arange(0, BLOCK_SIZE_M)

    matrix_ptr = matrix_ptr + (matrix_x[None, :] + matrix_y[:, None])

    val = tl.load(vec_ptr).to(tl.float32)
    matrix = tl.load(matrix_ptr).to(tl.float32)
    # TODO: USE in grouped gemm
    tl.store(out_ptr, tl.sum(val[:, None] * matrix, 0), mask=mask)

In [5]:
def get_config():
    config_file_path = "/home/ubuntu/vllm/benchmarks/kernels/config_vec.json"
    if os.path.exists(config_file_path):
        with open(config_file_path) as f:
            return {int(key): val for key, val in json.load(f).items()}

In [6]:
@triton.jit
def mat_vec_kernel(
    vec_ptr,
    matrix_ptr,
    out_ptr,
    vec_stridex,
    matrix_stridey,
    matrix_stridex,
    out_stridex,
    K,
    BLOCK_SIZE_M: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE_M
    offsets = block_start + tl.arange(0, BLOCK_SIZE_M)
    mask = offsets < K
    # vec_ptr = vec_ptr + vec_stridex * tl.arange(0, BLOCK_SIZE_M)
    #vec_ptr = tl.reshape(vec_ptr, (BLOCK_SIZE_M, 1))

    # out_ptr = out_ptr + out_stridex * tl.arange(0, BLOCK_SIZE_M)
    #out_ptr = tl.reshape(out_ptr, (BLOCK_SIZE_M, 1))

    matrix_x = (matrix_stridex * tl.arange(0, BLOCK_SIZE_M))
    matrix_y = (matrix_stridey * tl.arange(0, BLOCK_SIZE_M))

    matrix_ptrs = block_start + matrix_ptr + (matrix_x[None, :] + matrix_y[:, None])

    mask = tl.arange(0, BLOCK_SIZE_M) < K

    val = tl.load(vec_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
    matrix = tl.load(matrix_ptrs, mask=(offsets[:, None] < K) & (offsets[None, :] < K), other=0.0).to(tl.float32)
    #tl.store(out_ptr, tl.sum(val[:, None] * matrix, 0))
    tl.store(out_ptr + offsets, tl.sum(val[:, None] *matrix, axis=0), mask=mask)


In [7]:
K = 32
dtype =  torch.float16
    
vec = torch.randn((1,K), dtype=dtype, device="cuda")
out = torch.zeros((1,K), dtype=dtype, device="cuda")
matrix = torch.randn((K, K), dtype=dtype, device="cuda")
configs = get_config()
config = configs[1]

#grid = (1,)
grid = lambda meta: (triton.cdiv(K, meta['BLOCK_SIZE_M']), )
mat_vec_kernel[grid](
        vec,
        matrix,
        out,
        vec.stride(1),
        matrix.stride(0),
        matrix.stride(1),
        out.stride(1),
        K,
        32,
)

In [8]:
out

tensor([[  4.0156,  -5.7656,  -7.4961,   6.6211,   6.9141,  -9.2969,  -6.7461,
          -9.2266,  -7.1992,   9.8359,   0.7764,  -3.6289,  10.2266,  -7.3320,
           0.5991,   6.5781, -14.7656, -13.7969, -13.2578,  -5.3945,   1.4502,
          -0.6709,   5.2383,   2.3086,   6.6055,  11.4375,  -2.3613,   1.4336,
           4.4961,   6.7305,  -5.8594,  -0.3542]], device='cuda:0',
       dtype=torch.float16)

In [9]:
ref = torch.matmul(vec, matrix)
ref

tensor([[  4.0156,  -5.7656,  -7.4961,   6.6211,   6.9141,  -9.2969,  -6.7461,
          -9.2266,  -7.1992,   9.8359,   0.7764,  -3.6289,  10.2266,  -7.3320,
           0.5991,   6.5781, -14.7656, -13.7969, -13.2578,  -5.3945,   1.4502,
          -0.6709,   5.2383,   2.3086,   6.6055,  11.4375,  -2.3613,   1.4336,
           4.4961,   6.7305,  -5.8594,  -0.3542]], device='cuda:0',
       dtype=torch.float16)

In [10]:
ref = torch.matmul(vec, matrix)
assert torch.allclose(out, ref, atol=1e-2, rtol=1e-2)

In [13]:
# New method over k
@triton.jit
def mat_vec_mul_kernel(
    A, # Pointer to the matrix A (M x K)
    B, # Pointer to the vctor B (K x 1 or just K elements)
    C, # Pointer to the output vector C (M x 1m or just M elements)
    M, # Number of rows in A
    K, # Number of columns in A (and elements in B)
    BLOCK_SIZE_M: tl.constexpr, # Block size for M dimension
    BLOCK_SIZE_K: tl.constexpr, # Block size for K dimension
):
    # Determine the program ID (row index for the output of C)
    pid_m = tl.program_id(axis=0)

    #Initialize accumulator for the current row of C
    acc = tl.zeros((BLOCK_SIZE_M,), dtype=tl.float32)

    # Loop over the K dimension (columns of A) and elements of B
    for k_idx in range(0, K, BLOCK_SIZE_K):
        # Load block of A
        offs_am = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
        offs_ak = k_idx + tl.arange(0, BLOCK_SIZE_K)
        a_ptrs = A + (offs_am[:, None] * K + offs_ak[None,:])
        a = tl.load(a_ptrs, mask=(offs_am[:,None] < M) & (offs_ak[None, :] < K), other=0.0)

        # Load block of B
        offs_bk = k_idx + tl.arange(0, BLOCK_SIZE_K)
        b_ptrs = B + offs_bk
        b = tl.load(b_ptrs, mask=offs_bk < K, other = 0.0)

        # Perform dot product and accumualte
        acc += tl.sum(a*b[None, :], axis=1)
    
    # Store the result in the output vector C
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    c_ptrs = C + offs_cm
    tl.store(c_ptrs, acc, mask=offs_cm < M)

In [14]:
def mat_vec_mul(A,B):
    M, K = A.shape
    # assert B.shape == (K,), "Vector B must have K elements"
    C = torch.empty((M,), device=A.device, dtype=A.dtype)

    # Choose block sizes (tune these for optimal performance)
    BLOCK_SIZE_M = 16
    BLOCK_SIZE_K = 16

    grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE_M']),)
    mat_vec_mul_kernel[grid](A, B, C, M, K, BLOCK_SIZE_M, BLOCK_SIZE_K)
    return C

In [15]:
N = 1
M  =128
K = 128
A = torch.randn((M,K), dtype=dtype, device="cuda")
B = torch.randn((K,N), dtype=dtype, device="cuda")
C = torch.zeros((M, N), dtype=dtype, device="cuda")

C = mat_vec_mul(A,B)

In [16]:
C

tensor([  0.9976, -11.9375, -21.7656, -10.5000,  -5.6719,   6.7656, -12.0000,
          6.3477,  -8.0469,  -2.4922,  18.1094,  16.8906,   3.3770, -13.4219,
        -22.3750,   8.6406,  12.9219,  -0.8252, -20.5625,  13.1641,  10.4062,
        -13.8594,  10.5859, -15.9531,  -2.6641,  -6.4453,  -4.5938,   5.1055,
          6.7578, -19.2188,  -9.7266,   6.8086,  11.9922, -28.9062, -10.8203,
         36.0312,   2.0664,  24.0156,  -2.3477,   5.1680,  12.3750,  -2.3281,
          3.6211,   3.7305, -14.7969, -13.1094,  15.6875,  -0.4043, -10.0703,
        -22.8125,   1.0039,   8.3672,  -3.6914,   2.0664,  -9.4062,  -1.4297,
         -2.9902, -22.5469,   5.1484,  -2.7891,  13.7344, -15.0469,   5.1250,
          3.0547,   4.1250, -36.9375,  -5.8164, -11.8750,  10.3281,  -0.0928,
         -6.9141,  26.7344,   3.2734, -12.3750,  -7.8320,  12.0938,  14.6172,
         -8.8438,   3.8223,   4.6953, -16.6094,  10.2500,  -8.1953,   1.7773,
         -4.6016,  16.2344,  10.9688,  12.1094, -15.5938, -11.17

In [17]:
ref = torch.matmul(A, B)
ref

tensor([[  1.0049],
        [-11.9297],
        [-21.7500],
        [-10.4922],
        [ -5.6680],
        [  6.7617],
        [-12.0000],
        [  6.3438],
        [ -8.0469],
        [ -2.4922],
        [ 18.1094],
        [ 16.8906],
        [  3.3867],
        [-13.4141],
        [-22.3594],
        [  8.6328],
        [ 12.9219],
        [ -0.8218],
        [-20.5781],
        [ 13.1719],
        [ 10.4062],
        [-13.8594],
        [ 10.5938],
        [-15.9531],
        [ -2.6641],
        [ -6.4492],
        [ -4.6016],
        [  5.1016],
        [  6.7617],
        [-19.2188],
        [ -9.7344],
        [  6.8086],
        [ 11.9922],
        [-28.9062],
        [-10.8281],
        [ 36.0312],
        [  2.0703],
        [ 24.0312],
        [ -2.3516],
        [  5.1680],
        [ 12.3828],
        [ -2.3223],
        [  3.6211],
        [  3.7383],
        [-14.8047],
        [-13.1172],
        [ 15.6875],
        [ -0.4065],
        [-10.0625],
        [-22.8281],


In [18]:
ref = torch.matmul(A, B)
torch.max(abs(C-ref))/torch.max(ref)

tensor(2.0254, device='cuda:0', dtype=torch.float16)

In [19]:
@triton.jit
def row_vector_matrix_multiply_kernel(
    x_ptr, # Pointer to the row vector (1 x K)
    A_ptr, # Pointer to the matrix (K, N)
    y_ptr, # Pointer to the output vector (1 x N)
    K, # Number of columns in x and rows in A
    N, # Number of columns in A and in y
    BLOCK_SIZE_K: tl.constexpr, # Block size for K dimension
    BLOCK_SIZE_N: tl.constexpr, # Block size for N dimension
):
    # Get the program ID for the N dimension
    pid_n = tl.program_id(axis=0)

    # Calculate offsets for the K dimension (for loading x and A)
    offs_k = tl.arange(0, BLOCK_SIZE_K)

    # Initialize accumulator for the output vector block
    acc = tl.zeros((BLOCK_SIZE_N,), dtype=tl.float32)

    # Loop over the K dimenseion (columns of x rows of A)
    for k_start in range(0, K, BLOCK_SIZE_K):
        # Load block of row vector x
        # This will be a 1D vector of BLOCK_SIZE_K
        x_block_ptr = x_ptr + k_start + offs_k
        x_block = tl.load(x_block_ptr, mask  = offs_k < K, other = 0.0)

        # Load block of matrix A
        # This will be a 2D block of size (BLOCK_SIZE_K, BLOCK_SIZE_N)
        # Note the stride for A to access elements efficiently
        A_block_ptr = A_ptr + (k_start + offs_k[:, None]) * N + (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N))[None, :]
        A_block = tl.load(A_block_ptr, mask = (offs_k[:,None] < K) & ((pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N))[None, :] < N), other=0.0)

        # Perform block multiplication and accumulation
        acc += tl.sum(x_block[:, None] * A_block, axis=0) # tl.dot(x_block, A_block)

    # Store the accumulated results into the outptu vector y
    # This will be a 1d vector of size BLOCK_SIZE_N
    y_offs = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    tl.store(y_ptr + y_offs, acc, mask = y_offs < N)

# Wrapper fucntion to launch the kernel
def row_vector_times_matrix(x, A):
    # Ensure inputs are contiguous
    x = x.contiguous()
    A = A.contiguous()

    # Get dimensions
    assert x.shape[0] == 1, "x must be a row vector"
    assert x.shape[1] == A.shape[0], "Inner dimension must match"
    K = x.shape[1]
    N = A.shape[1]

    # Output vector
    y = torch.empty((1,N), device=x.device, dtype=x.dtype)

    # Configure block sizes (these are crucial for performance might need tuning)
    BLOCK_SIZE_K = 32
    BLOCK_SIZE_N = 32

    # Calculate grid dimensions based on N dimension
    grid = (triton.cdiv(N, BLOCK_SIZE_N),)

    # Launch the kernel
    row_vector_matrix_multiply_kernel[grid](
        x, A, y, K, N, BLOCK_SIZE_K, BLOCK_SIZE_N
    )
    return y

In [35]:
K = 5192
N = 2048

x = torch.randn((1,K), device='cuda', dtype=torch.float16)
A = torch.randn((K,N), device='cuda', dtype=torch.float16)

y = row_vector_times_matrix(x, A)

y_torch = torch.matmul(x,A)


In [36]:
y

tensor([[ -6.2500, -31.1719,  -5.6914,  ..., 127.0625, -66.8750,  62.7188]],
       device='cuda:0', dtype=torch.float16)

In [37]:
y_torch

tensor([[ -6.2461, -31.1562,  -5.6797,  ..., 127.0000, -66.8750,  62.6875]],
       device='cuda:0', dtype=torch.float16)

In [32]:
torch.max(abs(y - y_torch)/torch.max(y_torch))

In [33]:
tol = 1e-1
assert torch.allclose(y, y_torch, atol=tol, rtol=tol)

In [50]:
y.shape

torch.Size([1, 2048])

In [41]:
@triton.jit
def vec_add_kernel(x_ptr, # Pointer to first input vecotr
            y_ptr, # Pointer to second input vector
            output_ptr, # Pointer to output vecor
            n_elements, # size of vecotr
            BLOCK_SIZE_M: tl.constexpr): # Number of elements each program should process
      # NOTE: `constexpr` so it can be used as a shape value.
    # There are multiple programs processing different data. We identify which program we are here:
    pid = tl.program_id(0) # We use a 1D launch grid so axis is 0
    # This  program will process inputs that are offset from the initial data.
    # For instance, if you had a vector of length 256 and block_size of 64, the program
    # would each access the elements [0:64, 64:128, 128:192, 192: 256]
    # Note offsets is a list of pointers
    block_start = pid * BLOCK_SIZE_M
    offsets = block_start + tl.arange(0, BLOCK_SIZE_M)
    # Create mask to guard memory operation against out-of-bounds accesses
    mask  = offsets < n_elements

    # Load x and y from DRAM masking out any extra elements in case the nput is not a multiple of the block size
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y

    # Write x+ y back to DRAm
    tl.store(output_ptr + offsets, output, mask=mask)

In [42]:
DEVICE = triton.runtime.driver.active.get_active_torch_device()
def add(x: torch.Tensor, y: torch.Tensor):
    # We need to prallocate the output
    output = torch.empty_like(x)
    assert x.device == DEVICE and y.device == DEVICE and output.device == DEVICE
    n_elements = output.numel()
    # The SPMD launch grid denotes the number of kernel instances that run in parallel.
    # It is analogous to CUDA launch grids. It can be either Tuple[int], or Callable(metaparameters) -> Tuple[int].
    # In this case, we use a 1D grid where the size is the number of blocks:
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE_M']), )
    # NOTE:
    # - Each torch.tensor object is implicitly coverted into a pointer to its first element.
    # - triton.jit'ed functioncs can be indexed with a laucnh grid to obtain a callable GPU kernel
    # - Don't forget to pass meta-params as keyword args
    vec_add_kernel[grid](x,y,output,n_elements, BLOCK_SIZE_M=1024)
    # we return a handle to z but since `torch.cuda.synchronize()` hasn't been called, the kernel is still
    # running asynchronously at this point.
    return output

In [44]:
torch.manual_seed(0)
size = 98432
x = torch.rand(size, device=DEVICE)
y = torch.rand(size, device=DEVICE)
output_torch = x + y
output_triton = add(x, y)
print(output_torch)
print(output_triton)
print(f'The maximum difference between torch and triton is '
      f'{torch.max(torch.abs(output_torch - output_triton))}')
    # running asynchronously at this point.

tensor([1.3713, 1.3076, 0.4940,  ..., 0.6724, 1.2141, 0.9733], device='cuda:0')
tensor([1.3713, 1.3076, 0.4940,  ..., 0.6724, 1.2141, 0.9733], device='cuda:0')
The maximum difference between torch and triton is 0.0
